# Chatbot de vendas: grafo, conversas, métricas e testes

Este notebook lê o banco de conversas (`conversas.db`), o mesmo que o app grava. Para ver dados novos, converse com o bot (`streamlit run app.py`) e rode o notebook de novo.

1. **Grafo:** os nós do LangGraph e como a conversa passa entre eles.
2. **Conversas:** a lista de conversas salvas e o histórico completo de uma delas.
3. **Métricas:** os números calculados por `metricas.py`, os mesmos da página Métricas do app.
4. **Testes:** as regras em código (`test_bot.py`) e as conversas simuladas com o modelo (`cenarios.py`).

In [ ]:
import pandas as pd
from IPython.display import Image, display

import metricas
from bot import BANCO, ETAPAS, construir_grafo, etapa

banco = BANCO  # troque pelo caminho de outro arquivo para analisar outro banco
grafo = construir_grafo(banco=banco)
conversas = metricas.carregar_conversas(banco)
df = metricas.tabela(conversas)
print(f"{len(conversas)} conversas salvas em {banco}")

## 1. Grafo

- **agente:** o LLM conversa com o cliente e decide se chama uma tool. A resposta passa pela moderação antes de chegar ao cliente.
- **tools:** executa `listar_produtos`, `escolher_produto` e `registrar_cliente`, que validam os dados antes de mudar o estado.
- Depois das tools, `rota()` olha a etapa calculada a partir do estado e decide: volta ao **agente**, gera a **proposta** (montada em Python) ou faz o **handoff** para um especialista.

A imagem é gerada pelo serviço mermaid.ink, que recebe só a estrutura do grafo (os nomes dos nós), nenhum dado de conversa. Sem internet, a célula mostra o diagrama em texto Mermaid.

In [ ]:
try:
    display(Image(grafo.get_graph().draw_mermaid_png()))
except Exception as erro:  # sem acesso ao mermaid.ink
    print(f"Não foi possível gerar a imagem ({erro}). Diagrama em Mermaid:\n")
    print(grafo.get_graph().draw_mermaid())

## 2. Conversas

Conversas salvas, da mais recente para a mais antiga.

In [ ]:
if df.empty:
    print("Ainda não há conversas. Converse com o bot pelo app e rode o notebook de novo.")
else:
    display(df.assign(etapa=df.etapa.map(ETAPAS))[["id", "atualizada", "etapa", "itens", "total", "mensagens_cliente"]])

Histórico completo de uma conversa: mensagens do cliente e do bot, chamadas de tools (→) com os argumentos, retornos das tools (←), respostas barradas pela moderação e a trajetória de etapas pela qual a conversa passou.

In [ ]:
def mostrar_conversa(conversa_id: str):
    config = {"configurable": {"thread_id": conversa_id}}
    nomes_tools = {}
    for m in grafo.get_state(config).values.get("messages", []):
        if m.type == "human":
            print(f"\nCliente: {m.text}")
        elif m.type == "tool":
            print(f"   ← {nomes_tools.get(m.tool_call_id, 'tool')}: {m.text[:200]}")
        else:
            for chamada in m.tool_calls:
                nomes_tools[chamada["id"]] = chamada["name"]
                print(f"   → {chamada['name']}({chamada['args']})")
            if moderacao := m.additional_kwargs.get("moderacao"):
                print(f"   [barrada pela moderação, etapa '{moderacao['etapa']}'] {moderacao['original']}")
            if m.text:
                print(f"Bot: {m.text}")

    # Uma etapa por checkpoint, do mais antigo ao mais recente, sem repetições seguidas.
    etapas = [etapa(s.values) for s in reversed(list(grafo.get_state_history(config))) if s.values]
    trajetoria = [e for i, e in enumerate(etapas) if i == 0 or e != etapas[i - 1]]
    print("\nTrajetória:", " → ".join(ETAPAS[e] for e in trajetoria))


if conversas:
    mostrar_conversa(conversas[0]["id"])  # a mais recente; troque pelo id de outra conversa da tabela acima

## 3. Métricas

Calculadas em código a partir do histórico, sem LLM: o mesmo histórico sempre gera os mesmos números.

In [ ]:
if not df.empty:
    display(pd.Series(metricas.indicadores(df), name="valor").to_frame())

**Funil de vendas:** quantas conversas chegaram a cada etapa.

In [ ]:
if not df.empty:
    display(metricas.funil(df).style.bar(subset="conversas", color="#9ec5e8", vmin=0).hide(axis="index"))

**Problemas encontrados:** regras que valem para qualquer conversa (total igual ao catálogo, proposta nunca acima do CAP, transferência acima do CAP, nenhum link inventado), e-mail ou telefone que o cliente digitou e não foram registrados, e dado registrado que o cliente não digitou.

In [ ]:
if not df.empty:
    problemas = metricas.contar_problemas(df)
    display(problemas) if not problemas.empty else print("Nenhuma regra violada.")

**Respostas barradas pela moderação**, com o texto original, para revisar falsos positivos.

In [ ]:
barradas = metricas.moderadas(conversas)
display(barradas) if not barradas.empty else print("Nenhuma resposta barrada.")

**Todas as colunas por conversa.**

In [ ]:
if not df.empty:
    display(df.drop(columns=["abrir"]))

## 4. Testes

Os resultados registrados nesta seção foram obtidos com o **Qwen3 8B** (local, via Ollama). As células usam o modelo definido em `MODELO` no `.env`; com outro modelo, como o Claude Haiku 5.5, os números dos cenários podem mudar.

**Regras em código**, sem LLM: rodam em segundos e dão sempre o mesmo resultado, com qualquer modelo. `ExitCode.OK` no fim quer dizer que todos passaram.

In [ ]:
import pytest

pytest.main(["-q", "test_bot.py"])

**Conversas simuladas** com o modelo de verdade: cada cenário roda 3 vezes e a linha mostra em quantas o bot chegou ao resultado esperado. A primeira linha mostra o modelo usado. Leva alguns minutos com o modelo local. Para mais rodadas, troque o 3.

In [ ]:
%run cenarios.py 3

**Resultado com o Qwen3 8B**, 5 rodadas por cenário (19/10/2026):

| Cenário | Acertos |
|---|---:|
| Conversa mínima do desafio | 0/5 |
| Todos os dados numa mensagem, fora de ordem | 5/5 |
| Só o e-mail: o bot pede o celular | 5/5 |
| E-mail inválido e depois corrigido | 5/5 |
| Troca de produto | 0/5 |
| Acima do CAP: transfere para um humano | 5/5 |

**Limitações encontradas com o Qwen3 8B**

- **Produto fora do carrinho:** na conversa mínima, o cliente diz "Eu queria o NovoPhone X12" e o modelo responde com o preço e pede os dados, mas não chama `escolher_produto`. Sem produto no carrinho a proposta não sai, e o bot chega a dizer que o carrinho está pronto. Quando o cliente diz "Quero comprar o NovoPhone X12", o modelo chama a tool.
- **Troca vira soma:** "troca pelo GameBox X" adiciona o GameBox sem tirar o produto anterior (para isso o modelo precisaria chamar `escolher_produto` com quantidade 0). O total passa do CAP e o atendimento vai para um humano sem necessidade.
- **Texto interno no chat:** em testes manuais, o modelo às vezes repetiu para o cliente o retorno da tool ("Dados registrados. Carrinho: ...").

**O que testei para a conversa mínima**, 10 rodadas cada:

| Tentativa | Acertos | Observação |
|---|---:|---|
| Aviso no retorno de `registrar_cliente` quando o carrinho está vazio | 1/10 | O modelo ignora o aviso ou pede ao cliente para confirmar o produto |
| Tools marcadas no esqueleto da conversa do prompt | 4/10 | Mas o cenário do CAP caiu de 5/5 para 0/5 |
| Mais um passo 3 com exemplos ("eu queria o X") | 3/10 | |
| Mais as conversas de tom sem os preços | 0/10 | O modelo passou a dizer "custa R$ 00,00" |
| Modo thinking ligado, prompt original | 10/10 | Mediana de 16 s por mensagem, contra 1,4 s sem thinking, e respostas de mais de 1 minuto |

Nenhum ajuste de prompt ficou no código. O thinking resolve, mas deixa a resposta cerca de 11 vezes mais lenta, então o bot continua sem thinking.